# Módulo VI — Machine Learning

Código do livro *Ciência de Dados com Python* (Howard Roatti). Explicações no livro; repositório: https://github.com/howardroatti/roatti-ciencia-de-dados-python

## Introdução ao *Machine Learning*
### Machine Learning: O Que É?
### Quais as Formas de Aprendizagem?
### Machine Learning: Onde Se Aplica?
### Fluxo de Trabalho do Aprendizado de Máquina
## *Featuring Engineering*
### Tipos de Dados
### Preparação dos Dados
#### Dados Categóricos

In [ ]:
import os, sys
import pandas as pd
# Gerador de dados fictícios do livro (codigo/src/dados_sinteticos_m6.py)
sys.path.insert(0, "../src" if os.path.isdir("../src") else "codigo/src")
from dados_sinteticos_m6 import gerar_imoveis
# Geração do DataSet fictício de imóveis (1.500 registros, semente fixa)
df_imoveis = gerar_imoveis(n=1500, seed=42)
# Análise dos primeiros dados
df_imoveis.head(n=10)
# Separação dos Atributos que Passarão Pelo Processo
object_cols = list(df_imoveis.select_dtypes(exclude='number').columns)
print("Variáveis Categóricas:")
print(object_cols)
# Aplicando o LabelEncoder
from sklearn.preprocessing import LabelEncoder
# Criando uma Cópia para Evitar Mudanças nos Dados Originais
df_imoveis_le = df_imoveis.copy()
# Aplicação do Label Encoder para cada coluna listada como categórica
for col in object_cols:
    le = LabelEncoder().fit(df_imoveis_le[col])
    df_imoveis_le[col] = le.transform(df_imoveis_le[col])

In [ ]:
# Assuma que até aqui já foi aplicado a codificação pelo LabelEncoder
# Então siga com os passos a seguir
from sklearn.preprocessing import OneHotEncoder
# Aplicação do one-hot encoder para cada atributo categórico
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
oh = OH_encoder.fit(df_imoveis_le[object_cols])
df_imoveis_cols = pd.DataFrame(oh.transform(df_imoveis_le[object_cols]))
# O one-hot encoder remove os índices das amostras, portanto, coloque-as de volta
df_imoveis_cols.index = df_imoveis_le.index
# Remova os atributos categóricos
numerical_df_imoveis = df_imoveis_le.drop(object_cols, axis=1)
# Crie um novo Data Frame com os atributos numéricos e concatene com as colunas do One-Hot Encoder
df_imoveis_oh = pd.concat([numerical_df_imoveis, df_imoveis_cols], axis=1)
df_imoveis_oh.columns = df_imoveis_oh.columns.astype(str)

#### Dados Numéricos

In [ ]:
import numpy as np
# Listando os atributos numéricos
numerical_cols = list(df_imoveis.select_dtypes(include='number').columns)
print("Variáveis Numéricas:")
print(numerical_cols)
# Importando o Método de Padronização
from sklearn.preprocessing import StandardScaler
# Criando uma cópia dos dados para aplicar a padronização
df_imoveis_std = df_imoveis.copy()
# Aplicação da Padronização nos Atributos Numéricos
for column in numerical_cols:
    transformer = StandardScaler()
    # Para esse tipo de operação, você deverá mudar o formato dos dados
    values = np.array(df_imoveis_std[column]).reshape(-1, 1)
    df_imoveis_std[column] = transformer.fit_transform(values)

#### Dados Ausentes
### Seleção de Características e Redução de Dimensionalidade

In [ ]:
# Importando o Método de Normalização (min-max)
from sklearn.preprocessing import MinMaxScaler
# Criando uma cópia dos dados para aplicar a normalização
df_imoveis_norm = df_imoveis.copy()
# Aplicação da Normalização nos Atributos Numéricos
for column in numerical_cols:
    transformer = MinMaxScaler()
    # Para esse tipo de operação, você deverá mudar o formato dos dados
    values = np.array(df_imoveis_norm[column]).reshape(-1, 1)
    df_imoveis_norm[column] = transformer.fit_transform(values)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Atributo alvo: imóvel caro (preço acima da mediana) ou não
df_imoveis['caro'] = (df_imoveis['preco'] > df_imoveis['preco'].median()).astype(int)
cols = ['quartos', 'distancia_km', 'cep', 'banheiros', 'vagas', 'idade_anos',
        'area_terreno', 'latitude', 'longitude', 'imoveis_bairro']
X, y = df_imoveis[cols], df_imoveis['caro']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
# Padronização (ajustada somente no conjunto de treino)
scaler = StandardScaler().fit(X_train)
std_X_train = pd.DataFrame(scaler.transform(X_train), columns=cols, index=X_train.index)
# Treinamento do Random Forest
modelo = RandomForestClassifier(n_estimators=200, random_state=42)
modelo.fit(std_X_train, y_train)

In [ ]:
from matplotlib import pyplot as plt
%matplotlib inline
# Recupera a importância dos coeficientes obtidos pelo modelo
importances = modelo.feature_importances_
# Calcula o desvio padrão das importâncias em cada estimador
std = np.std([tree.feature_importances_ for tree in modelo.estimators_],
axis=0)
# Recupera os índices dos atributos ordenados pela importância
indices = np.argsort(importances)[::-1]
# Lista a importância das características
for f in range(X.shape[1]):
    print("%d. feature %s (%.3f%%)" % (f + 1, std_X_train.columns[indices[f]], importances[indices[f]]*100))

In [ ]:
# Plota um gráfico com as importâncias e o desvio padrão
plt.figure(figsize=(12,10))
plt.title("Feature importances")
plt.bar(std_X_train.columns[indices],
importances[indices],
color="r",
yerr=std[indices],
align="center")
plt.xticks(std_X_train.columns[indices],
rotation=45)
plt.xlim([-1, X.shape[1]])
plt.show()

In [ ]:
threshold_per_feature = 0.05
features_per_feature = []
# Lista as Características que Atendem ao Threshold definido
for f in range(X.shape[1]):
    if importances[indices[f]] >= threshold_per_feature:
        print("%d. feature %s (%.3f%%)" % (f + 1, std_X_train.columns[indices[f]], importances[indices[f]]*100))
        features_per_feature.append(std_X_train.columns[indices[f]])

In [ ]:
print()
print("Features Based on Accumulative Threshold")
print(features_per_feature)

In [ ]:
threshold_accumulative = 0.95
total = 0
features_accumulative = []
# Lista as Características que Atendem ao Threshold definido
for f in range(X.shape[1]):
    print("%d. feature %s (%.3f%%)" % (f + 1, std_X_train.columns[indices[f]], importances[indices[f]]*100))
    total += importances[indices[f]]
    features_accumulative.append(
        std_X_train.columns[indices[f]])
    if total >= threshold_accumulative:
        break

In [ ]:
print()
print("Total: %.3f%% \t Threshold Accumulative: %.3f%%" % (total*100, threshold_accumulative*100))
print()
print("Features Based on Accumulative Threshold")
print(features_accumulative)

#### Redução de Dimensionalidade

In [ ]:
from sklearn.decomposition import PCA
# nesse exemplo serão gerados apenas dois componentes principais
pca = PCA(n_components = 2)
components = pca.fit_transform(X)

In [ ]:
''' nesse exemplos serão gerados a quantidade de componentes necessários para que se tenha 95% de variância. '''
pca = PCA(n_components = 0.95)
components = pca.fit_transform(X)

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt
# Carrega o conjunto de dados e padroniza as 30 características
cancer = load_breast_cancer()
x = StandardScaler().fit_transform(cancer.data)
# No scikit-learn, 0 = maligno e 1 = benigno
breast_dataset = pd.DataFrame({'label': pd.Series(cancer.target).map({0: 'Malignant', 1: 'Benign'})})
pca_breast = PCA(n_components=2)
principalComponents_breast = pca_breast.fit_transform(x)
principal_breast_Df = pd.DataFrame(data = principalComponents_breast
, columns = ['principal component 1', 'principal component 2'])
%matplotlib inline
plt.figure()
plt.figure(figsize=(10,10))
plt.xticks(fontsize=12)
plt.yticks(fontsize=14)
plt.xlabel('Principal Component - 1',fontsize=20)
plt.ylabel('Principal Component - 2',fontsize=20)
plt.title("Principal Component Analysis of Breast Cancer Dataset",fontsize=20)
targets = ['Benign', 'Malignant']
colors = ['r', 'g']
for target, color in zip(targets,colors):
    indicesToKeep = breast_dataset['label'] == target
    plt.scatter(principal_breast_Df.loc[indicesToKeep, 'principal component 1']
    , principal_breast_Df.loc[indicesToKeep, 'principal component 2'], c = color, s = 50)
    plt.legend(targets,prop={'size': 15})

## Algoritmos
### Algoritmos de Aprendizagem Não Supervisionada
#### k-Means

In [ ]:
from sklearn.datasets import make_blobs
# Gerando os dados a partir do make_blobs
features, true_labels = make_blobs(
n_samples=200,
centers=3,
cluster_std=2.75,
random_state=42
)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
# Reescalando os valores gerados para que outliers não interfiram no modelo
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
# Criando o k-Means
kmeans = KMeans(
init="random",
n_clusters=3,
n_init=10,
max_iter=300,
random_state=42
)
# Ensinando ao modelo como separar os dados
kmeans.fit(scaled_features)

In [ ]:
# Uma nova amostra, reescalada com o mesmo scaler usado no treinamento
nova_amostra = scaler.transform([[0.0, 0.0]])
kmeans.predict(nova_amostra)

In [ ]:
for i in range(3):
    term = "===Cluster %d===" % i
    print(term.replace(term, "\033[48;5;0m\033[38;5;226m {term} \033[0;0m".format(term=term)))
    for j in range(len(features)):
        if kmeans.labels_[j] == i:
            print(kmeans.labels_[j], ' ', features[j])

#### Clusterização Hierárquica

In [ ]:
# Bibliotecas necessárias para utilização do modelo
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import BaseEstimator, clone
from sklearn.utils.metaestimators import available_if

In [ ]:
def _classificador_tem(metodo):
    # Expõe o método apenas se o classificador interno o possui
    return lambda self: hasattr(self.classifier_, metodo)


class InductiveClusterer(BaseEstimator):
    def __init__(self, clusterer, classifier):
        self.clusterer = clusterer
        self.classifier = classifier

    def fit(self, X, y=None):
        self.clusterer_ = clone(self.clusterer)
        self.classifier_ = clone(self.classifier)
        y = self.clusterer_.fit_predict(X)
        self.classifier_.fit(X, y)
        return self

    @available_if(_classificador_tem('predict'))
    def predict(self, X):
        return self.classifier_.predict(X)

    @available_if(_classificador_tem('decision_function'))
    def decision_function(self, X):
        return self.classifier_.decision_function(X)

In [ ]:
clusterer = AgglomerativeClustering(n_clusters=3, metric='euclidean', linkage='ward')
clusterer.fit(scaled_features)

In [ ]:
# Cria o classificador
classifier = RandomForestClassifier(random_state=42)
''' Cria o classificador indutivo, que irá transferir a aprendizagem do clusterizador para o classificador'''
inductive_learner = InductiveClusterer(clusterer, classifier).fit(scaled_features)
# Realiza a predição de qual cluster a nova amostra será associada
inductive_learner.predict(nova_amostra)

In [ ]:
for i in range(3):
    term = "===Cluster %d===" % i
    print(term.replace(term, "\033[48;5;0m\033[38;5;226m {term} \033[0;0m".format(term=term)))
    for j in range(len(features)):
        if clusterer.labels_[j] == i:
            print(clusterer.labels_[j], ' ', features[j])

### Algoritmos de Aprendizagem Supervisionada
#### Naive Bayes

In [ ]:
# Bibliotecas utilizadas
import os, sys
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
# Gerador de dados fictícios do livro (codigo/src/dados_sinteticos_m6.py)
sys.path.insert(0, "../src" if os.path.isdir("../src") else "codigo/src")
from dados_sinteticos_m6 import gerar_emprestimos
# Geração dos dados fictícios de empréstimos (5.000 registros, semente fixa)
loan_data = gerar_emprestimos(n=5000, seed=42)
# Conversão dos dados para categorias
loan_data.outcome = loan_data.outcome.astype('category')
loan_data.outcome = loan_data.outcome.cat.reorder_categories(['paid off', 'default'])
loan_data.purpose_ = loan_data.purpose_.astype('category')
loan_data.home_ = loan_data.home_.astype('category')
loan_data.emp_len_ = loan_data.emp_len_.astype('category')
# Separação dos atributos utilizados para treinamento
predictors = ['purpose_', 'home_', 'emp_len_']
# Atributo alvo, o que queremos predizer
outcome = 'outcome'
# Aplicação de One-Hot-Encoder
X = pd.get_dummies(loan_data[predictors], prefix='', prefix_sep='')
y = loan_data[outcome]
# Criação e treinamento do modelo
naive_model = MultinomialNB(alpha=0.01, fit_prior=True)
naive_model.fit(X, y)
# Separação da amostra para teste
new_loan = X.loc[146:146, :]
print('predicted class: ', naive_model.predict(new_loan)[0])
# Probabilidades geradas para a amostra a ser classificaada
probabilities = pd.DataFrame(naive_model.predict_proba(new_loan),
columns=naive_model.classes_)
print('predicted probabilities',)
probabilities

#### k-Nearest Neighbors -- kNN

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import neighbors, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do modelo com k=3 e posterior treinamento
modelo = neighbors.KNeighborsClassifier(n_neighbors=3)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

In [ ]:
# Análise da taxa de erro obtida ao avaliar valores para k
error = []
# Calculating error for K values between 1 and 40
for i in range(1, 40):
    knn = neighbors.KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    pred_i = knn.predict(X_test)
    error.append(np.mean(pred_i != y_test))

#### Rocchio

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import neighbors, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do modelo com métrica de similaridade distância Manhattan
modelo = neighbors.NearestCentroid(metric='manhattan')
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

#### Support Vector Machine -- SVM

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import svm, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do modelo SVM linear e posterior treinamento
modelo = svm.LinearSVC(penalty='l2', dual=False, tol=1e-3)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

#### Decision Tree -- DT

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import tree, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação da árvore de decisão (critério: entropia) e posterior treinamento
modelo = tree.DecisionTreeClassifier(random_state=13, criterion='entropy',
min_impurity_decrease=0.003)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

#### Ensemble: Bagging (Random Forest) e Boosting (XGboost)

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import ensemble, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do Random Forest com 50 árvores e posterior treinamento
modelo = ensemble.RandomForestClassifier(n_estimators=50, random_state=13, oob_score=True)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import datasets
from xgboost import XGBClassifier
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do modelo XGBoost (3 classes: objetivo multiclasse com probabilidades)
modelo = XGBClassifier(objective='multi:softprob', subsample=.63, eval_metric='merror', random_state=13)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

In [ ]:
# Biblioteca para combinar os parâmetros no loop
from itertools import product
# seleção de índices aleatórios (fold de 0 a 4) para cada amostra, com semente fixa
rng = np.random.default_rng(42)
idx = rng.integers(0, 5, size=len(X))
# lista que irá armazenar os resultados e valores de parâmetros
error = []
# Loop que irá combinar os valores dos parâmetros eta, max_depth e n_estimators
for eta, max_depth, n in product([0.1, 0.5, 0.9], [3, 6, 9, 12], list(range(10, 250, 10))):
    # criação do modelo com as configurações dos parâmetros atuais
    xgb = XGBClassifier(objective='multi:softprob', n_estimators=n,
    max_depth=max_depth, learning_rate=eta,
    eval_metric='merror', n_jobs=2, random_state=13)
    # lista para armazenar a taxa de erro de cada fold da validação cruzada
    cv_error = []
    for k in range(5):
        # índice do fold a ser utilizado
        fold_idx = idx == k
        # Separação de treino e teste
        train_X = pd.DataFrame(X).loc[~fold_idx]; train_y = y[~fold_idx]
        valid_X = pd.DataFrame(X).loc[fold_idx]; valid_y = y[fold_idx]
        # Treino do XGBoost
        xgb.fit(train_X, train_y)
        # Predição do modelo atual (classe com maior probabilidade)
        pred = xgb.predict(valid_X)
        # Taxa de erro do modelo atual neste fold
        cv_error.append(np.mean(pred != valid_y))
    # média dos erros dos 5 folds para esta combinação de parâmetros
    error.append({
    'eta': eta,
    'max_depth': max_depth,
    'n_estimators': n,
    'avg_error': np.mean(cv_error)
    })
errors = pd.DataFrame(error)
errors

#### Regressão Linear

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
import pandas as pd
diabetes = load_diabetes()
df_diabetes = pd.DataFrame(diabetes.data, columns = diabetes.feature_names)
df_diabetes['target'] = diabetes.target
from sklearn.model_selection import train_test_split
X = df_diabetes.loc[:,(df_diabetes.columns != 'target')]
y = df_diabetes['target']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.33, random_state=42)
modelo = LinearRegression()
modelo.fit(X_train, y_train)
predictions = modelo.predict(X_valid)

In [ ]:
from sklearn.decomposition import PCA
pca_diabetes = PCA(n_components=2)
principalComponents_diabetes = pca_diabetes.fit_transform(X_valid)
principal_diabetes_Df = pd.DataFrame(data = principalComponents_diabetes
, columns = ['principal component 1', 'principal component 2'])
principal_diabetes_Df['y'] = y_valid
principal_diabetes_Df['predicts'] = predictions

#### Regressão Estocástica Descida do Gradiente

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import SGDRegressor
import pandas as pd
diabetes = load_diabetes()
df_diabetes = pd.DataFrame(diabetes.data, columns = diabetes.feature_names)
df_diabetes['target'] = diabetes.target
from sklearn.model_selection import train_test_split
X = df_diabetes.loc[:,(df_diabetes.columns != 'target')]
y = df_diabetes['target']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.33, random_state=42)
modelo = SGDRegressor(eta0=0.1, learning_rate='invscaling', penalty='l2', shuffle=False)
modelo.fit(X_train, y_train)
predictions = modelo.predict(X_valid)

#### Regressão Logística

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import linear_model, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do modelo de Regressão Logística e posterior treinamento
modelo = linear_model.LogisticRegression()
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

#### Redes Neurais Artificiais

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import linear_model, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do Perceptron e posterior treinamento
modelo = linear_model.Perceptron()
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

In [ ]:
# Módulos do Scikit Learn utilizados
from sklearn import neural_network, datasets
# Carregamento do conjunto de dados Iris Flower
iris = datasets.load_iris()
# Separando os dados em variáveis independentes e variável dependente
X = iris.data
y = iris.target
# separa o conjunto de dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=13)
# Criação do Perceptron Multicamadas (MLP) e posterior treinamento
modelo = neural_network. MLPClassifier(hidden_layer_sizes=(32,),
random_state=13, solver='sgd',
max_iter=10000, activation='relu',
learning_rate='adaptive', alpha=0.01)
modelo.fit(X_train, y_train)
# Prevendo o tipo de flor para o conjunto de dados de teste
pred = modelo.predict(X_test)

## Avaliando Modelos
### Avaliação dos Modelos de Machine Learning

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
# The mean squared error
print('Mean square error: %.2f' % mean_squared_error(y_valid, predictions))
# The mean absolute error
print('Mean absolute error: %.2f' % mean_absolute_error(y_valid, predictions))
# The coefficient of determination: 1 is perfect prediction
print('Coefficient of determination (R²): %.2f%%' % (r2_score(y_valid, predictions) * 100))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import roc_curve, accuracy_score, roc_auc_score
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
# Problema binário: no scikit-learn 0 = maligno e 1 = benigno
cancer = load_breast_cancer()
lista_classes = list(cancer.target_names)
nome_classe_positiva = 1
X_tr, X_te, y_train, y_test = train_test_split(cancer.data, cancer.target, test_size=0.3,
                                               stratify=cancer.target, random_state=13)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_train)
predicted = clf.predict(X_te)
# Probabilidade da classe positiva (necessária para a curva ROC e para a AUC)
scores = clf.predict_proba(X_te)[:, 1]
# Exibe as métricas Recall, Precision e F1
print(metrics.classification_report(y_test, predicted, target_names=lista_classes))
# Exibe a acurácia
print(accuracy_score(y_test, predicted))
# Computa AUC
auc = roc_auc_score(y_test, scores)
print(auc)
# Recupera o recall e precision para plotar a curva ROC
fpr, tpr, thresholds = roc_curve(y_test, scores, pos_label=nome_classe_positiva)
roc_df = pd.DataFrame({'recall': tpr, 'specificity': 1 - fpr})
# Plota a curva ROC
ax = roc_df.plot(x='specificity', y='recall', figsize=(10, 10), legend=False)
ax.set_ylim(0, 1)
ax.set_xlim(1, 0)
ax.plot((1, 0), (0, 1))
ax.set_xlabel('specificity')
ax.set_ylabel('recall')
ax.fill_between(roc_df.specificity, 0, roc_df.recall, alpha=0.3)
ax.text(.5,.5,'AUC: %.3f' % (auc*100),
bbox={'facecolor':'white','alpha':1,'edgecolor':'none','pad':1},
ha='center', va='center')
plt.tight_layout()
plt.show()

## Fazendo *Deploy* de Modelos em Produção
### Serializando o Modelo de Machine Learning

In [ ]:
import os, pickle
destino = './static/model/model.pkl'
os.makedirs(os.path.dirname(destino), exist_ok=True)
with open(destino, 'wb') as f:
    # Usa o protocolo com maior capacidade de compactação possível para serializar
    pickle.dump(modelo, f, pickle.HIGHEST_PROTOCOL)

In [ ]:
with open(destino, 'rb') as f:
    modelo = pickle.load(f)

### Criando uma API para o Modelo de Machine Learning

In [ ]:
import sys, os.path
# Diretório da aplicação (onde ficam app.py, static/ e templates/)
application_dir = os.path.dirname(os.path.abspath(globals().get('__file__', 'app.py')))
sys.path.append(application_dir)

In [ ]:
# Opcional: requer o Flask (pip install flask); o servidor só roda fora do notebook
from flask import Flask, request, render_template
import xgboost
import numpy as np
from pickle import load

In [ ]:
app = Flask(__name__)

# Carrega o modelo uma única vez, quando a aplicação é iniciada
with open(os.path.join(application_dir, "static", "model", "model.pkl"), 'rb') as f:
    app.predictor = load(f)


@app.route("/")
def index():
    return render_template('index.html', pred=0)


@app.route('/predict', methods=['POST'])
def predict():
    data = [request.form['spatial_length'],
            request.form['spatial_width'],
            request.form['petal_length'],
            request.form['petal_width']]
    data = np.array([np.asarray(data, dtype=float)]).reshape(1, -1)
    predictions = app.predictor.predict(data)
    print('INFO Predictions: ({}) ==> {}'.format(data, predictions))
    classes = ['Setosa', 'Versicolor', 'Virginica']
    class_ = "The sample ({}) is {}".format(data[0], classes[predictions[0]])
    return render_template('index.html', pred=class_)


def main():
    """Run the app."""
    app.run(host='0.0.0.0', port=8000, debug=False)


if __name__ == '__main__':
    main()

## Exercícios